## autograd

### 第 1 步：requires_grad=True（开启求导）

In [1]:
import torch

In [9]:
a = torch.tensor([[1.0],[2.0]],requires_grad= True)
print(a)

tensor([[1.],
        [2.]], requires_grad=True)


### 第 2 步：backward()

In [11]:
b = torch.tensor(2.0,requires_grad=True)
c = b ** 2
c.backward()

print(b.grad)

b = torch.tensor(3.0,requires_grad=True)
c = b ** 3 + 2 *b
c.backward()
print(b)
print(b.grad)


tensor(4.)
tensor(3., requires_grad=True)
tensor(29.)


梯度存储在：叶子节点 tensor 的 .grad 属性里

在 PyTorch 的计算图中，“叶子节点”通常指那些**由你直接创建**、而不是通过运算得到的张量。它们通常是模型的参数（如权重 weight 和偏置 bias）或需要求导的输入数据。

In [13]:
#常见坑
x = torch.tensor(2.0, requires_grad=True)

y = x * 2
z = y * 3

z.backward()
print(x.grad)
print(y.grad)  # ❌ None

tensor(6.)
None


E:\TEMP\Temp\ipykernel_14124\4001868354.py:9: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\build\aten\src\ATen/core/TensorBody.h:494.)
  print(y.grad)  # ❌ None


计算目标：计算 z 相对于 x 的梯度。根据链式法则，dz/dx = (dz/dy) * (dy/dx) = 3 * 2 = 6。
存储位置：PyTorch 计算出这个梯度 6 之后，会把它存放在叶子节点 x 的 .grad 属性里，即 x.grad 会变成 tensor(6.)。
中间结果：y 只是一个中间计算步骤。为了节省内存，PyTorch 默认不会为这些非叶子节点保留梯度。因此，y.grad 的值是 None。

In [14]:
x = torch.tensor(2.0, requires_grad=True)
y = x * 2
# 【关键步骤】告诉 PyTorch 保留 y 的梯度
y.retain_grad()
z = y * 3
z.backward()
print(f"x.grad: {x.grad}") # 输出: tensor(6.)
print(f"y.grad: {y.grad}") # 输出: tensor(3.)，现在不再是 None 了！

x.grad: 6.0
y.grad: 3.0


In [20]:
x = torch.tensor(2.0, requires_grad=True)
y = x * 2
z = y + 3
z.backward()
print(x.grad)

tensor(2.)


### 第 3 步：detach()切断梯度

 实际用途：
* 在迁移学习或训练复杂模型时，我们常常希望只更新模型的一部分参数，而保持另一部分参数不变。
* GAN中固定生成器或者判别器
* 将tensor安全地转换成numpy

In [27]:
x = torch.tensor(2.0, requires_grad=True)
y = x * 2
z = y.detach()
w = z * 3
w.backward()
print(x.grad)

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

### 梯度累积（重要的坑）

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
y.backward()

y = x ** 2
y.backward()
print(x.grad)

tensor(8.)


🧠 为什么？
👉 因为：
grad = 4 + 4
👉 梯度会累加！！

In [31]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
y.backward()
x.grad.zero_()
y = x ** 2
y.backward()
print(x.grad)

tensor(4.)


### 手写loss + backward
y =wx 根据链式法则得到最终结果

In [32]:
w = torch.tensor(3.0,requires_grad=True)
x = torch.tensor(2.0)

y_pred = w*x
y_true = torch.tensor(5.0)

loss = (y_pred - y_true)**2
loss.backward()
print(w.grad)


tensor(4.)


In [34]:
w = torch.tensor(3.0,requires_grad=True)
x = torch.tensor(2.0)
y_pred = w*x
y_true = torch.tensor(5.0)
loss = (w*x - y_true)**2
loss.backward()
print(w.grad)


tensor(4.)
